# ClinicSense – Notebook 03: Predictive Modelling
**B.Sc. Data Science Honours | Student: Kadam Akanksha Dinkar**

This notebook covers:
- Phase 5: Predictive Modelling
- Logistic Regression, Random Forest, XGBoost, LightGBM, Stacking Ensemble
---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.models import (build_logistic_regression, build_random_forest,
                         build_xgboost, build_lightgbm, build_stacking_ensemble,
                         save_model)
from src.evaluation import (
    compute_metrics, plot_confusion_matrix,
    plot_roc_curves, plot_pr_curves, print_report, comparison_table
)
from src.config import FIGURES_DIR, TARGET

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Setup complete.')

Setup complete.


## 1. Load Splits

In [2]:
data_dir = os.path.join('..', 'data')

X_train, X_test, y_train, y_test = joblib.load(
    os.path.join(data_dir, 'splits.joblib'))
X_train_smote, y_train_smote = joblib.load(
    os.path.join(data_dir, 'smote_splits.joblib'))

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'X_train_smote: {X_train_smote.shape}')
print(f'Train no-show rate: {y_train.mean():.3f}')
print(f'Test  no-show rate: {y_test.mean():.3f}')

# class imbalance ratio for XGBoost
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'\nScale pos weight for XGBoost: {pos_weight:.2f}')

X_train: (57567, 21) | X_test: (14392, 21)
X_train_smote: (82298, 21)
Train no-show rate: 0.285
Test  no-show rate: 0.285

Scale pos weight for XGBoost: 2.51


## 2. Model 1: Logistic Regression (Baseline)

In [3]:
print('Training Logistic Regression...')
lr = build_logistic_regression()
lr.fit(X_train, y_train)   # uses class_weight='balanced'

lr_pred  = lr.predict(X_test)
lr_prob  = lr.predict_proba(X_test)[:, 1]

print_report(y_test, lr_pred, 'Logistic Regression')
plot_confusion_matrix(y_test, lr_pred, 'Logistic Regression')
lr_metrics = compute_metrics(y_test, lr_pred, lr_prob, 'Logistic Regression')
print('\nMetrics:', lr_metrics)

save_model(lr, 'logistic_regression')

Training Logistic Regression...



  Classification Report – Logistic Regression
              precision    recall  f1-score   support

        Show       0.77      0.58      0.66     10288
     No-Show       0.35      0.57      0.43      4104

    accuracy                           0.57     14392
   macro avg       0.56      0.57      0.55     14392
weighted avg       0.65      0.57      0.59     14392




Metrics: {'model': 'Logistic Regression', 'accuracy': 0.5745553085047248, 'precision': 0.34944071588366893, 'recall': 0.570906432748538, 'f1': 0.4335276158756592, 'roc_auc': 0.6082585546968553, 'pr_auc': 0.38549693731021195, 'brier_score': 0.24022340451846136}
[Models] Saved → C:\Users\LENOVO\OneDrive\Desktop\ClinicSense\outputs\models\logistic_regression.joblib


'C:\\Users\\LENOVO\\OneDrive\\Desktop\\ClinicSense\\outputs\\models\\logistic_regression.joblib'

## 3. Model 2: Random Forest

In [4]:
print('Training Random Forest...')
rf = build_random_forest()
rf.fit(X_train_smote, y_train_smote)

rf_pred  = rf.predict(X_test)
rf_prob  = rf.predict_proba(X_test)[:, 1]

print_report(y_test, rf_pred, 'Random Forest')
plot_confusion_matrix(y_test, rf_pred, 'Random Forest')
rf_metrics = compute_metrics(y_test, rf_pred, rf_prob, 'Random Forest')
print('\nMetrics:', rf_metrics)

save_model(rf, 'random_forest')

Training Random Forest...



  Classification Report – Random Forest
              precision    recall  f1-score   support

        Show       0.75      0.70      0.72     10288
     No-Show       0.36      0.43      0.39      4104

    accuracy                           0.62     14392
   macro avg       0.56      0.56      0.56     14392
weighted avg       0.64      0.62      0.63     14392




Metrics: {'model': 'Random Forest', 'accuracy': 0.6200667037242913, 'precision': 0.36070261437908496, 'recall': 0.43031189083820665, 'f1': 0.39244444444444443, 'roc_auc': 0.5979054687002628, 'pr_auc': 0.36918463128169543, 'brier_score': 0.22848943632047847}


[Models] Saved → C:\Users\LENOVO\OneDrive\Desktop\ClinicSense\outputs\models\random_forest.joblib


'C:\\Users\\LENOVO\\OneDrive\\Desktop\\ClinicSense\\outputs\\models\\random_forest.joblib'

## 4. Model 3: XGBoost

In [5]:
print('Training XGBoost...')
xgb = build_xgboost(scale_pos_weight=pos_weight)
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100)

xgb_pred  = xgb.predict(X_test)
xgb_prob  = xgb.predict_proba(X_test)[:, 1]

print_report(y_test, xgb_pred, 'XGBoost')
plot_confusion_matrix(y_test, xgb_pred, 'XGBoost')
xgb_metrics = compute_metrics(y_test, xgb_pred, xgb_prob, 'XGBoost')
print('\nMetrics:', xgb_metrics)

save_model(xgb, 'xgboost')

Training XGBoost...


[0]	validation_0-auc:0.60331


[100]	validation_0-auc:0.63267


[199]	validation_0-auc:0.63560



  Classification Report – XGBoost
              precision    recall  f1-score   support

        Show       0.78      0.63      0.70     10288
     No-Show       0.38      0.56      0.45      4104

    accuracy                           0.61     14392
   macro avg       0.58      0.60      0.58     14392
weighted avg       0.67      0.61      0.63     14392




Metrics: {'model': 'XGBoost', 'accuracy': 0.6129794330183436, 'precision': 0.3789230260984473, 'recall': 0.5589668615984406, 'f1': 0.45166371332939553, 'roc_auc': 0.6356044955003501, 'pr_auc': 0.4232701401515402, 'brier_score': 0.23159289804420038}
[Models] Saved → C:\Users\LENOVO\OneDrive\Desktop\ClinicSense\outputs\models\xgboost.joblib


'C:\\Users\\LENOVO\\OneDrive\\Desktop\\ClinicSense\\outputs\\models\\xgboost.joblib'

## 5. Model 4: LightGBM

In [6]:
print('Training LightGBM...')
lgbm = build_lightgbm()
lgbm.fit(X_train, y_train,
         eval_set=[(X_test, y_test)])

lgbm_pred  = lgbm.predict(X_test)
lgbm_prob  = lgbm.predict_proba(X_test)[:, 1]

print_report(y_test, lgbm_pred, 'LightGBM')
plot_confusion_matrix(y_test, lgbm_pred, 'LightGBM')
lgbm_metrics = compute_metrics(y_test, lgbm_pred, lgbm_prob, 'LightGBM')
print('\nMetrics:', lgbm_metrics)

save_model(lgbm, 'lightgbm')

Training LightGBM...



  Classification Report – LightGBM
              precision    recall  f1-score   support

        Show       0.78      0.65      0.71     10288
     No-Show       0.38      0.55      0.45      4104

    accuracy                           0.62     14392
   macro avg       0.58      0.60      0.58     14392
weighted avg       0.67      0.62      0.64     14392




Metrics: {'model': 'LightGBM', 'accuracy': 0.6193718732629239, 'precision': 0.38367761598374533, 'recall': 0.5521442495126706, 'f1': 0.45274725274725275, 'roc_auc': 0.6370204650888713, 'pr_auc': 0.42400510633816346, 'brier_score': 0.23006206110776897}
[Models] Saved → C:\Users\LENOVO\OneDrive\Desktop\ClinicSense\outputs\models\lightgbm.joblib


'C:\\Users\\LENOVO\\OneDrive\\Desktop\\ClinicSense\\outputs\\models\\lightgbm.joblib'

## 6. Model 5: Stacking Ensemble

In [7]:
print('Training Stacking Ensemble (this may take several minutes)...')
stack = build_stacking_ensemble(scale_pos_weight=pos_weight)
stack.fit(X_train_smote, y_train_smote)

stack_pred  = stack.predict(X_test)
stack_prob  = stack.predict_proba(X_test)[:, 1]

print_report(y_test, stack_pred, 'Stacking Ensemble')
plot_confusion_matrix(y_test, stack_pred, 'Stacking Ensemble')
stack_metrics = compute_metrics(y_test, stack_pred, stack_prob, 'Stacking Ensemble')
print('\nMetrics:', stack_metrics)

save_model(stack, 'stacking_ensemble')

Training Stacking Ensemble (this may take several minutes)...



  Classification Report – Stacking Ensemble
              precision    recall  f1-score   support

        Show       0.74      0.89      0.81     10288
     No-Show       0.44      0.22      0.30      4104

    accuracy                           0.70     14392
   macro avg       0.59      0.55      0.55     14392
weighted avg       0.65      0.70      0.66     14392




Metrics: {'model': 'Stacking Ensemble', 'accuracy': 0.6970539188438021, 'precision': 0.43852065321805955, 'recall': 0.22246588693957114, 'f1': 0.2951826705463951, 'roc_auc': 0.5922254020846786, 'pr_auc': 0.3906088434317275, 'brier_score': 0.20782040227204002}


[Models] Saved → C:\Users\LENOVO\OneDrive\Desktop\ClinicSense\outputs\models\stacking_ensemble.joblib


'C:\\Users\\LENOVO\\OneDrive\\Desktop\\ClinicSense\\outputs\\models\\stacking_ensemble.joblib'

## 7. Model Comparison – ROC and PR Curves

In [8]:
models_results = {
    'Logistic Regression': (y_test, lr_prob),
    'Random Forest':       (y_test, rf_prob),
    'XGBoost':             (y_test, xgb_prob),
    'LightGBM':            (y_test, lgbm_prob),
    'Stacking Ensemble':   (y_test, stack_prob),
}

plot_roc_curves(models_results)
plot_pr_curves(models_results)
print('ROC and PR curve plots saved.')

ROC and PR curve plots saved.


## 8. Comparison Table

In [9]:
all_metrics = [lr_metrics, rf_metrics, xgb_metrics, lgbm_metrics, stack_metrics]
comp_table = comparison_table(all_metrics)
print('\nModel Comparison Table:')
print('='*75)
print(comp_table.to_string())
print('='*75)

# Save comparison table
comp_table.to_csv(os.path.join('..', 'outputs', 'reports', 'model_comparison.csv'))
print('\nComparison table saved to outputs/reports/model_comparison.csv')

# Identify best model
best_model_name = comp_table['f1'].idxmax()
print(f'\nBest model by F1-score: {best_model_name}')

# Save all probabilities for later notebooks
probs_dict = {
    'y_test':   y_test.values,
    'lr_prob':  lr_prob,
    'rf_prob':  rf_prob,
    'xgb_prob': xgb_prob,
    'lgbm_prob':lgbm_prob,
    'stack_prob': stack_prob,
}
joblib.dump(probs_dict, os.path.join('..', 'data', 'model_probs.joblib'))
print('Model probabilities saved for downstream notebooks.')


Model Comparison Table:
                     accuracy  precision  recall      f1  roc_auc  pr_auc  brier_score
model                                                                                 
Logistic Regression    0.5746     0.3494  0.5709  0.4335   0.6083  0.3855       0.2402
Random Forest          0.6201     0.3607  0.4303  0.3924   0.5979  0.3692       0.2285
XGBoost                0.6130     0.3789  0.5590  0.4517   0.6356  0.4233       0.2316
LightGBM               0.6194     0.3837  0.5521  0.4527   0.6370  0.4240       0.2301
Stacking Ensemble      0.6971     0.4385  0.2225  0.2952   0.5922  0.3906       0.2078

Comparison table saved to outputs/reports/model_comparison.csv

Best model by F1-score: LightGBM
Model probabilities saved for downstream notebooks.


## 9. Summary

**Five models trained and compared:**
- Logistic Regression (interpretable baseline with StandardScaler + class_weight='balanced')
- Random Forest (300 trees, trained on SMOTE-balanced data)
- XGBoost (500 estimators, scale_pos_weight to handle imbalance)
- LightGBM (500 estimators, class_weight='balanced')
- Stacking Ensemble (LR + RF + XGB + LGBM → LR meta-learner)

**Key evaluation metrics used:** Precision, Recall, F1, ROC-AUC, PR-AUC, Brier Score.  
Accuracy alone is avoided due to the 80/20 class imbalance.

The best model will be tuned and calibrated in Notebook 04.